In [19]:
!pip install -q -U transformers datasets accelerate

In [20]:
import torch

print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No se detectó ninguna GPU.")

CUDA disponible: True
GPU: Tesla T4


In [21]:
from datasets import load_dataset, Dataset

# Cargar el conjunto de datos en modo streaming
datos = load_dataset(
    "igorshilov/tinystories_spanish",
    split="train",
    streaming=True
)

cuentos = []

# Seleccionar 3000 cuentos en español
for ejemplo in datos:
    texto = ejemplo["text_es"].strip()

    if texto and len(texto.split()) >= 30:
        cuentos.append(texto)

    if len(cuentos) >= 3000:
        break

print("Cantidad de cuentos:", len(cuentos))

# Crear el dataset
dataset = Dataset.from_dict({
    "text": cuentos
})

# Separar entrenamiento y validación
dataset = dataset.train_test_split(
    test_size=0.1,
    seed=42
)

print(dataset)

Cantidad de cuentos: 3000
DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2700
    })
    test: Dataset({
        features: ['text'],
        num_rows: 300
    })
})


In [22]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

modelo_nombre = "DeepESP/gpt2-spanish"

# Cargar tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    modelo_nombre
)

# Agregar un token especial para padding
tokenizer.add_special_tokens({
    "pad_token": "<|pad|>"
})

# Cargar modelo Transformer
model = AutoModelForCausalLM.from_pretrained(
    modelo_nombre
)

# Actualizar los embeddings por el nuevo token
model.resize_token_embeddings(len(tokenizer))

model.config.pad_token_id = tokenizer.pad_token_id

print("Modelo cargado correctamente")
print("Dispositivo:", "GPU" if torch.cuda.is_available() else "CPU")

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: DeepESP/gpt2-spanish
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo cargado correctamente
Dispositivo: GPU


In [23]:
def tokenizar(ejemplos):

    textos = [
        texto + tokenizer.eos_token
        for texto in ejemplos["text"]
    ]

    return tokenizer(
        textos,
        truncation=True,
        max_length=384
    )


dataset_tokenizado = dataset.map(
    tokenizar,
    batched=True,
    remove_columns=["text"]
)

print("Tokenización completada")

Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Tokenización completada


In [24]:
# Asegurar que el modelo utilice precisión FP32
model = model.float()

# Limpiar gradientes anteriores
for parameter in model.parameters():
    parameter.grad = None

print("Tipo de datos:", next(model.parameters()).dtype)
print("Modelo preparado para entrenamiento")

Tipo de datos: torch.float32
Modelo preparado para entrenamiento


In [25]:
from transformers import (
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)

# Preparar los datos
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Configuración del entrenamiento
training_args = TrainingArguments(
    output_dir="./transformer_cuentos",

    num_train_epochs=2,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=35,

    logging_steps=25,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    # Desactivar precisión mixta
    fp16=False,
    bf16=False,

    report_to="none"
)

# Crear el entrenador
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_tokenizado["train"],
    eval_dataset=dataset_tokenizado["test"],
    data_collator=data_collator,
    processing_class=tokenizer
)

print("¡Entrenador configurado correctamente!")

¡Entrenador configurado correctamente!


In [26]:
# Iniciar entrenamiento
resultado = trainer.train()

print("Entrenamiento finalizado")

# Guardar el modelo
trainer.save_model("./transformer_cuentos_final")
tokenizer.save_pretrained("./transformer_cuentos_final")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50257}.


Epoch,Training Loss,Validation Loss
1,2.151346,2.098817
2,1.935893,2.060282


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Entrenamiento finalizado


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./transformer_cuentos_final/tokenizer_config.json',
 './transformer_cuentos_final/tokenizer.json')

In [27]:
import math

resultados = trainer.evaluate()

print("Pérdida de validación:",
      resultados["eval_loss"])

perplejidad = math.exp(resultados["eval_loss"])

print("Perplejidad:",
      perplejidad)

Training Loss,Validation Loss,Epoch
1.935893,2.060282,2


Pérdida de validación: 2.0602824687957764
Perplejidad: 7.848186365001495


In [28]:
import torch

model.eval()

entrada = "Había una vez un pequeño dragón que"

inputs = tokenizer(
    entrada,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():

    logits = model(**inputs).logits[:, -1, :]

    probabilidades = torch.softmax(
        logits,
        dim=-1
    )

    valores, indices = torch.topk(
        probabilidades,
        k=5
    )

print("Posibles siguientes tokens:")

for token_id, probabilidad in zip(
    indices[0],
    valores[0]
):

    token = tokenizer.decode([token_id])

    print(
        repr(token),
        round(probabilidad.item(), 4)
    )

Posibles siguientes tokens:
' vivía' 0.4904
' le' 0.1097
' amaba' 0.0659
' quería' 0.0614
' tenía' 0.0563


In [29]:
entrada = (
    "Había una vez, en un bosque mágico, "
    "un pequeño zorro que encontró una llave."
)

inputs = tokenizer(
    entrada,
    return_tensors="pt"
).to(model.device)

salida = model.generate(
    **inputs,
    max_new_tokens=120,
    do_sample=True,
    temperature=0.8,
    top_p=0.92,
    repetition_penalty=1.15,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id
)

cuento = tokenizer.decode(
    salida[0],
    skip_special_tokens=True
)

print(cuento)

Había una vez, en un bosque mágico, un pequeño zorro que encontró una llave. La puso en su bolsillo y la abrió. Dentro de su bolsillo encontró una llave especial con muchos símbolos que podía usar para abrir el camino de vuelta a casa.

El zorro estaba muy emocionado de tener esa llave especial. Estaba tan emocionado por descubrirla que decidió jugar con ella también. La metió dentro del bolsillo e intentó abrirla, pero era demasiado pesada. De repente, escuchó algo extraño que lo asustó mucho. ¡Era la llave! El zorro pensó que era una llave mágica para él.

¡El zorro se sorprendió tanto al descubrirla! Miró a su alrededor y vio muchas cosas divertidas


In [30]:
model.eval()

prompts = [
    "Había una vez un pequeño dragón que vivía en una montaña.",
    "En un reino muy lejano, una princesa encontró un mapa secreto.",
    "Un día, un conejo decidió explorar un bosque desconocido."
]

for i, prompt in enumerate(prompts):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    salida = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.75,
        top_p=0.90,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    cuento = tokenizer.decode(
        salida[0],
        skip_special_tokens=True
    )

    print(f"\n--- CUENTO {i+1} ---\n")
    print(cuento)


--- CUENTO 1 ---

Había una vez un pequeño dragón que vivía en una montaña. Tenía muchos amigos, pero todos tenían sus cosas para hacer. Todos estaban muy emocionados porque iban a aprender algo nuevo.

Un día, el amigo de dragones vio al dragón por la cima del volcán. El dragón dijo: "¡Mira!" El dragón sonrió y dijo: "Está bien".

El dragón estaba tan emocionado y le agradeció al dragón antes de irse. El amigo dragón se despidió con la mano y voló lejos. El hermano dragón se fue volando, feliz de haber podido explorar la montaña.

--- CUENTO 2 ---

En un reino muy lejano, una princesa encontró un mapa secreto. Era una cámara mágica que brillaba con luz azul y brillante. En la pared de su habitación, había una foto de un tesoro mágico en el cielo nocturno.

La niña pequeña estaba tan emocionada por explorar el mundo mágico. Agarró su bolsa y comenzó a buscar entre las cosas maravillosas que veía. Pero cuando se acercó más al mapa, vio algo especial: era una cueva oscura con muchas cos